# MSP-Podcast Dataset Exploration

This notebook explores the manifest files generated by `scripts/create_msp_manifests.py`.

It assumes the following files exist:

```text
data/manifests/msp_podcast/
├── all.csv
├── train.csv
├── validation.csv
├── test.csv
└── label_schema.json
```

Inspects split sizes, label distributions, soft-label behavior, no-agreement samples, A/V/D attributes, secondary emotions, missing audio, and speaker metadata.

In [ ]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 200)

PROJECT_ROOT = Path.cwd().parent # Notebook has root from /notebooks always
assert PROJECT_ROOT.match("sailer_ser_reproduction")

MANIFEST_DIR = PROJECT_ROOT / "data" / "manifests" / "msp_podcast"
REPORT_DIR = PROJECT_ROOT / "outputs" / "msp_dataset_exploration"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MANIFEST_DIR:", MANIFEST_DIR)
print("REPORT_DIR:", REPORT_DIR)

In [ ]:
def read_manifest(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Run:\n"
            f"  python scripts/create_msp_manifests.py\n"
            f"from the project root first."
        )
    return pd.read_csv(path)

schema_path = MANIFEST_DIR / "label_schema.json"
if not schema_path.exists():
    raise FileNotFoundError(
        f"Missing {schema_path}. Run scripts/create_msp_manifests.py first."
    )

with schema_path.open("r", encoding="utf-8") as f:
    schema = json.load(f)

all_df = read_manifest(MANIFEST_DIR / "all.csv")
split_dfs = {
    "train": read_manifest(MANIFEST_DIR / "train.csv"),
    "validation": read_manifest(MANIFEST_DIR / "validation.csv"),
    "test": read_manifest(MANIFEST_DIR / "test.csv"),
}

primary_labels = schema.get("primary_labels") or [
    col.replace("target_", "").replace("_", " ").title()
    for col in all_df.columns
    if col.startswith("target_")
]
target_cols = schema.get("target_columns") or [
    f"target_{label.lower().replace(' ', '_')}" for label in primary_labels
]
target_cols = [col for col in target_cols if col in all_df.columns]

secondary_rate_cols = [
    col for col in all_df.columns
    if col.startswith("secondary_rate_")
]

print("Loaded manifests")
print("all.csv:", all_df.shape)
for split, df in split_dfs.items():
    print(f"{split}.csv:", df.shape)

print("\nPrimary labels:", primary_labels)
print("Target columns:", target_cols)
print("Secondary rate columns:", len(secondary_rate_cols))

## 1. Basic manifest overview

In [ ]:
display(all_df.head())

overview_rows = []
for split, df in {"all": all_df, **split_dfs}.items():
    row = {
        "split": split,
        "num_rows": len(df),
        "num_unique_files": df["filename"].nunique() if "filename" in df.columns else np.nan,
    }
    if "audio_exists" in df.columns:
        audio_exists = df["audio_exists"].astype(str).str.lower().isin(["true", "1", "yes"])
        row["audio_exists"] = int(audio_exists.sum())
        row["audio_missing"] = int((~audio_exists).sum())
    if "speaker_id" in df.columns:
        row["num_speakers"] = df["speaker_id"].replace("", np.nan).dropna().nunique()
    if "num_annotations" in df.columns:
        row["mean_num_annotations"] = df["num_annotations"].mean()
        row["min_num_annotations"] = df["num_annotations"].min()
        row["max_num_annotations"] = df["num_annotations"].max()
    overview_rows.append(row)

overview = pd.DataFrame(overview_rows)
display(overview)

overview.to_csv(REPORT_DIR / "overview.csv", index=False)

## 2. Sanity checks

These checks catch common issues before feature extraction or training.

In [ ]:
def target_sum_stats(df: pd.DataFrame, name: str) -> dict:
    if not target_cols:
        return {"split": name, "status": "no target columns found"}
    target_sum = df[target_cols].sum(axis=1)
    return {
        "split": name,
        "rows": len(df),
        "min_target_sum": float(target_sum.min()) if len(df) else np.nan,
        "mean_target_sum": float(target_sum.mean()) if len(df) else np.nan,
        "max_target_sum": float(target_sum.max()) if len(df) else np.nan,
        "num_not_close_to_1": int((~np.isclose(target_sum, 1.0, atol=1e-5)).sum()),
    }

checks = []
for split, df in {"all": all_df, **split_dfs}.items():
    checks.append(target_sum_stats(df, split))

checks_df = pd.DataFrame(checks)
display(checks_df)

if "filename" in all_df.columns:
    duplicated = all_df[all_df["filename"].duplicated(keep=False)].sort_values("filename")
    print("Duplicate filenames:", len(duplicated))
    if len(duplicated):
        display(duplicated[["filename", "split"]].head(20))

if "audio_exists" in all_df.columns:
    audio_exists = all_df["audio_exists"].astype(str).str.lower().isin(["true", "1", "yes"])
    missing_audio = all_df.loc[~audio_exists]
    print("Missing audio rows:", len(missing_audio))
    if len(missing_audio):
        display(missing_audio[["filename", "split", "audio_path"]].head(20))
        missing_audio.to_csv(REPORT_DIR / "missing_audio.csv", index=False)

checks_df.to_csv(REPORT_DIR / "sanity_checks.csv", index=False)

## 3. Split sizes

In [ ]:
split_counts = (
    all_df["split"].value_counts(dropna=False)
    .rename_axis("split")
    .reset_index(name="count")
)
display(split_counts)

plt.figure(figsize=(7, 4))
plt.bar(split_counts["split"].astype(str), split_counts["count"])
plt.title("Number of utterances per split")
plt.xlabel("Split")
plt.ylabel("Count")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 4. Soft primary-label distributions

`target_*` columns are the normalized worker primary-emotion distributions. These are the columns intended for soft-label training.

In [ ]:
def target_distribution(df: pd.DataFrame) -> pd.Series:
    if not target_cols:
        return pd.Series(dtype=float)
    totals = df[target_cols].sum(axis=0)
    total_mass = totals.sum()
    if total_mass == 0:
        return totals
    return totals / total_mass

dist_table = pd.DataFrame({
    split: target_distribution(df)
    for split, df in {"all": all_df, **split_dfs}.items()
}).T

# Cleaner column names for display.
dist_table.columns = [col.replace("target_", "").title() for col in dist_table.columns]
display(dist_table)

dist_table.to_csv(REPORT_DIR / "soft_label_distribution_by_split.csv")

plt.figure(figsize=(12, 5))
x = np.arange(len(dist_table.columns))
width = 0.8 / len(dist_table.index)

for i, split in enumerate(dist_table.index):
    plt.bar(x + i * width, dist_table.loc[split].values, width=width, label=split)

plt.title("Soft primary-label distribution by split")
plt.xlabel("Emotion")
plt.ylabel("Distribution mass")
plt.xticks(x + width * (len(dist_table.index) - 1) / 2, dist_table.columns, rotation=45, ha="right")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Consensus-label distributions and no-agreement samples

The consensus label is useful for reporting and hard-label metrics, but training should use the soft target distribution.

In [ ]:
if "consensus_label" in all_df.columns:
    consensus_table = pd.crosstab(all_df["split"], all_df["consensus_label"], dropna=False)
    display(consensus_table)
    consensus_table.to_csv(REPORT_DIR / "consensus_distribution_by_split.csv")

    consensus_all = all_df["consensus_label"].value_counts(dropna=False)
    plt.figure(figsize=(10, 4))
    plt.bar(consensus_all.index.astype(str), consensus_all.values)
    plt.title("Consensus-label distribution")
    plt.xlabel("Consensus label")
    plt.ylabel("Count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

if "has_consensus" in all_df.columns:
    no_consensus = all_df[~all_df["has_consensus"].astype(str).str.lower().isin(["true", "1", "yes"])]
elif "consensus_code" in all_df.columns:
    no_consensus = all_df[all_df["consensus_code"].astype(str).str.upper().eq("X")]
else:
    no_consensus = pd.DataFrame()

print("No-agreement / no-consensus rows:", len(no_consensus))
if len(no_consensus):
    cols = [c for c in ["filename", "split", "consensus_code", "consensus_label", "num_annotations", *target_cols] if c in no_consensus.columns]
    display(no_consensus[cols].head(20))
    no_consensus.to_csv(REPORT_DIR / "no_consensus_samples.csv", index=False)

## 6. Agreement, confidence, and soft-label entropy

High entropy means annotators were more spread across emotions. Low entropy means stronger agreement.

In [ ]:
def add_soft_label_diagnostics(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if not target_cols:
        return out

    targets = out[target_cols].fillna(0).to_numpy(dtype=float)
    target_sums = targets.sum(axis=1, keepdims=True)
    safe_targets = np.divide(targets, target_sums, out=np.zeros_like(targets), where=target_sums != 0)

    entropy = -(safe_targets * np.where(safe_targets > 0, np.log(safe_targets), 0)).sum(axis=1)
    max_entropy = math.log(len(target_cols)) if len(target_cols) > 1 else 1.0
    normalized_entropy = entropy / max_entropy

    top_idx = safe_targets.argmax(axis=1)
    top_prob = safe_targets.max(axis=1)
    top_label = [target_cols[i].replace("target_", "").title() for i in top_idx]

    out["soft_entropy"] = entropy
    out["soft_entropy_normalized"] = normalized_entropy
    out["top_soft_label"] = top_label
    out["top_soft_prob"] = top_prob
    return out

diagnostic_df = add_soft_label_diagnostics(all_df)

diag_summary = diagnostic_df.groupby("split").agg(
    mean_entropy=("soft_entropy", "mean"),
    mean_normalized_entropy=("soft_entropy_normalized", "mean"),
    mean_top_soft_prob=("top_soft_prob", "mean"),
    median_top_soft_prob=("top_soft_prob", "median"),
).reset_index()

display(diag_summary)
diag_summary.to_csv(REPORT_DIR / "soft_label_diagnostics_by_split.csv", index=False)

plt.figure(figsize=(8, 4))
for split in sorted(diagnostic_df["split"].dropna().unique()):
    values = diagnostic_df.loc[diagnostic_df["split"].eq(split), "top_soft_prob"].dropna()
    plt.hist(values, bins=20, alpha=0.5, label=split)
plt.title("Top soft-label probability")
plt.xlabel("Largest target probability")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
for split in sorted(diagnostic_df["split"].dropna().unique()):
    values = diagnostic_df.loc[diagnostic_df["split"].eq(split), "soft_entropy_normalized"].dropna()
    plt.hist(values, bins=20, alpha=0.5, label=split)
plt.title("Normalized soft-label entropy")
plt.xlabel("Entropy / log(num_classes)")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Majority and minority classes

The SAILER paper treats Neutral, Happy, Sad, and Angry as majority classes; the remaining emotions are minority classes. This section computes the soft-label mass for those two groups.

In [ ]:
majority_labels = {"neutral", "happy", "sad", "angry"}
majority_cols = [col for col in target_cols if col.replace("target_", "").lower() in majority_labels]
minority_cols = [col for col in target_cols if col not in majority_cols]

majority_minority_rows = []
for split, df in {"all": all_df, **split_dfs}.items():
    total_mass = df[target_cols].sum().sum() if target_cols else 0
    majority_mass = df[majority_cols].sum().sum() if majority_cols else 0
    minority_mass = df[minority_cols].sum().sum() if minority_cols else 0

    majority_minority_rows.append({
        "split": split,
        "majority_mass": majority_mass,
        "minority_mass": minority_mass,
        "majority_fraction": majority_mass / total_mass if total_mass else np.nan,
        "minority_fraction": minority_mass / total_mass if total_mass else np.nan,
    })

majority_minority_df = pd.DataFrame(majority_minority_rows)
display(majority_minority_df)
majority_minority_df.to_csv(REPORT_DIR / "majority_minority_soft_mass.csv", index=False)

plt.figure(figsize=(7, 4))
x = np.arange(len(majority_minority_df))
plt.bar(x, majority_minority_df["majority_fraction"], label="majority")
plt.bar(x, majority_minority_df["minority_fraction"], bottom=majority_minority_df["majority_fraction"], label="minority")
plt.title("Majority/minority soft-label mass by split")
plt.xlabel("Split")
plt.ylabel("Fraction")
plt.xticks(x, majority_minority_df["split"])
plt.legend()
plt.tight_layout()
plt.show()

## 8. Arousal, valence, and dominance

This section summarizes consensus A/V/D values and worker-mean A/V/D values, if present.

In [ ]:
avd_cols = [
    "consensus_arousal", "consensus_valence", "consensus_dominance",
    "mean_worker_arousal", "mean_worker_valence", "mean_worker_dominance",
]
avd_cols = [col for col in avd_cols if col in all_df.columns]

if not avd_cols:
    print("No A/V/D columns found.")
else:
    avd_summary = all_df.groupby("split")[avd_cols].agg(["mean", "std", "min", "max"])
    display(avd_summary)
    avd_summary.to_csv(REPORT_DIR / "avd_summary_by_split.csv")

    for col in avd_cols:
        plt.figure(figsize=(7, 4))
        for split in sorted(all_df["split"].dropna().unique()):
            values = pd.to_numeric(all_df.loc[all_df["split"].eq(split), col], errors="coerce").dropna()
            plt.hist(values, bins=14, alpha=0.5, label=split)
        plt.title(col)
        plt.xlabel("Score")
        plt.ylabel("Count")
        plt.legend()
        plt.tight_layout()
        plt.show()

## 9. Secondary emotion rates

Secondary labels are multi-label. The rates indicate the fraction of annotators who selected each secondary emotion.

In [ ]:
if not secondary_rate_cols:
    print("No secondary-rate columns found.")
else:
    secondary_mean = pd.DataFrame({
        split: df[secondary_rate_cols].mean(axis=0)
        for split, df in {"all": all_df, **split_dfs}.items()
    }).T
    secondary_mean.columns = [
        col.replace("secondary_rate_", "").replace("_", " ").title()
        for col in secondary_mean.columns
    ]
    display(secondary_mean)
    secondary_mean.to_csv(REPORT_DIR / "secondary_rate_means_by_split.csv")

    sec_all = secondary_mean.loc["all"].sort_values(ascending=False)
    plt.figure(figsize=(12, 4))
    plt.bar(sec_all.index, sec_all.values)
    plt.title("Mean secondary-emotion selection rate")
    plt.xlabel("Secondary emotion")
    plt.ylabel("Mean rate")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 10. Speaker and gender metadata

This section is only meaningful if `Speaker_ids.txt` was available when the manifests were created.

In [ ]:
if "speaker_id" not in all_df.columns:
    print("No speaker_id column found.")
else:
    speaker_df = all_df.copy()
    speaker_df["speaker_id_clean"] = speaker_df["speaker_id"].replace("", np.nan)

    speaker_counts = speaker_df.groupby("split")["speaker_id_clean"].nunique(dropna=True).reset_index(name="num_speakers")
    display(speaker_counts)
    speaker_counts.to_csv(REPORT_DIR / "speaker_counts_by_split.csv", index=False)

    utterances_per_speaker = (
        speaker_df.dropna(subset=["speaker_id_clean"])
        .groupby("speaker_id_clean")
        .size()
        .sort_values(ascending=False)
        .rename("num_utterances")
        .reset_index()
    )
    display(utterances_per_speaker.head(20))
    utterances_per_speaker.to_csv(REPORT_DIR / "utterances_per_speaker.csv", index=False)

    plt.figure(figsize=(8, 4))
    plt.hist(utterances_per_speaker["num_utterances"], bins=40)
    plt.title("Utterances per speaker")
    plt.xlabel("Number of utterances")
    plt.ylabel("Number of speakers")
    plt.tight_layout()
    plt.show()

if "speaker_gender" in all_df.columns:
    gender_table = pd.crosstab(all_df["split"], all_df["speaker_gender"].replace("", np.nan), dropna=False)
    display(gender_table)
    gender_table.to_csv(REPORT_DIR / "speaker_gender_by_split.csv")

## 11. Example subsets for inspection

Useful views for qualitative checks before modeling.

In [ ]:
view_cols = [
    "filename", "split", "audio_path", "consensus_code", "consensus_label",
    "num_annotations", "top_soft_label", "top_soft_prob", "soft_entropy_normalized",
    *target_cols,
]
view_cols = [col for col in view_cols if col in diagnostic_df.columns]

print("Most ambiguous samples by soft-label entropy")
display(
    diagnostic_df.sort_values("soft_entropy_normalized", ascending=False)[view_cols].head(20)
)

print("Highest-confidence samples")
display(
    diagnostic_df.sort_values("top_soft_prob", ascending=False)[view_cols].head(20)
)

if minority_cols:
    tmp = diagnostic_df.copy()
    tmp["minority_mass"] = tmp[minority_cols].sum(axis=1)
    print("Samples with largest minority-class target mass")
    display(
        tmp.sort_values("minority_mass", ascending=False)[
            [col for col in ["filename", "split", "consensus_label", "minority_mass", *target_cols] if col in tmp.columns]
        ].head(20)
    )

## 12. Save enriched exploration table

This writes a copy of `all.csv` plus diagnostic columns such as entropy and top soft-label probability.

In [ ]:
enriched_path = REPORT_DIR / "all_with_soft_label_diagnostics.csv"
diagnostic_df.to_csv(enriched_path, index=False)
print("Saved:", enriched_path)
print("Other summary files were saved under:", REPORT_DIR)

In [ ]:
enriched_df = pd.read_csv(enriched_path)

print(f"Shape: {enriched_df.shape[0]:,} rows × {enriched_df.shape[1]:,} columns")
display(enriched_df.head(20))